In [1]:
import sys
import asyncio

# Fix for Windows issues in Jupyter notebooks
if sys.platform == "win32":
    # 1. Use ProactorEventLoop for subprocess support
    if not isinstance(asyncio.get_event_loop_policy(), asyncio.WindowsProactorEventLoopPolicy):
        asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())

    # 2. Redirect stderr to avoid fileno() error when launching MCP servers
    if "ipykernel" in sys.modules:
        sys.stderr = sys.__stderr__

C:\Users\user\AppData\Local\Temp\ipykernel_11772\3336491994.py:7: DeprecationWarning: 'asyncio.get_event_loop_policy' is deprecated and slated for removal in Python 3.16
  if not isinstance(asyncio.get_event_loop_policy(), asyncio.WindowsProactorEventLoopPolicy):
C:\Users\user\AppData\Local\Temp\ipykernel_11772\3336491994.py:7: DeprecationWarning: 'asyncio.WindowsProactorEventLoopPolicy' is deprecated and slated for removal in Python 3.16
  if not isinstance(asyncio.get_event_loop_policy(), asyncio.WindowsProactorEventLoopPolicy):


In [4]:
import os

from dotenv import load_dotenv
from langchain_mcp_adapters.client import MultiServerMCPClient

load_dotenv()
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage


# 自定义模型信息
model = init_chat_model(
    model = os.getenv("CHAT_MODEL"),
    model_provider="openai",
    api_key=os.getenv("CHAT_API_KEY"),
    base_url=os.getenv("BASE_URL"),
    temperature=1.2,
    extra_body={
        "thinking": {
            "type": "enabled"
        },
        "reasoning_effort": "max"
    }
)

# 定义工具
## 用mcp获取工具
### 定义mcp client
client = MultiServerMCPClient({
    "time-mcp": {
        "transport": "stdio",
        "args": [
            "-y",
            "time-mcp"
        ],
        "command": "npx",
        "env": {"VOLTA_HOME": "${VOLTA_HOME}"}
    }
})

### 用client拉取工具
tools = await client.get_tools()

agent = create_agent(
    model=model,
    tools=tools
)

messages = await agent.ainvoke({
    "messages": [HumanMessage(content="现在几点了")]
})
from rich import print as fprint
fprint(messages)

{
    'messages': [
        HumanMessage(
            content='现在几点了',
            additional_kwargs={},
            response_metadata={},
            id='fa9a8705-7cd3-4b88-a024-599ab3b04107'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 79,
                    'prompt_tokens': 961,
                    'total_tokens': 1040,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': None,
                        'audio_tokens': None,
                        'reasoning_tokens': 15,
                        'rejected_prediction_tokens': None,
                        'text_tokens': None
                    },
                    'prompt_tokens_details': {
                        'audio_tokens': None,
                        'cache_write_tokens': 0,
                        'cached_tokens': 896,
                        'image_tokens': None,
                        'text_tokens': None
                    }
                },
                'model_provider': 'openai',
                'model_name': 'group/deepseek-flash',
                'system_fingerprint': None,
                'id': 'chatcmpl-02179473850738696188f7c040816dd3fe5310871d7143a949ec6',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--01a1dec0-4c02-75b4-96bb-231379c9495f-0',
            tool_calls=[
                {
                    'name': 'current_time',
                    'args': {'format': 'YYYY-MM-DD HH:mm:ss', 'timezone': 'Asia/Shanghai'},
                    'id': 'call_00_ufu1zhae3hiq8jo7htg447lb',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 961,
                'output_tokens': 79,
                'total_tokens': 1040,
                'input_token_details': {'cache_read': 896, 'cache_creation': 0},
                'output_token_details': {'reasoning': 15}
            }
        ),
        ToolMessage(
            content=[
                {
                    'type': 'text',
                    'text': 'Current UTC time is 2026-11-15 10:28:30, and the time in Asia/Shanghai is 2026-11-15 
18:28:30.',
                    'id': 'lc_0a2ca4c7-489e-482a-8ce5-4b5ad105c468'
                }
            ],
            name='current_time',
            id='ef90f12f-0b0c-42ba-97c9-ff4e9d61c504',
            tool_call_id='call_00_ufu1zhae3hiq8jo7htg447lb'
        ),
        AIMessage(
            content='现在是 **2026年11月15日 
18:28:30**（北京时间，UTC+8）。\n\n如果你想知道其他时区的时间，也可以告诉我。',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 34,
                    'prompt_tokens': 1078,
                    'total_tokens': 1112,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': None,
                        'audio_tokens': None,
                        'reasoning_tokens': 0,
                        'rejected_prediction_tokens': None,
                        'text_tokens': None
                    },
                    'prompt_tokens_details': {
                        'audio_tokens': None,
                        'cache_write_tokens': 0,
                        'cached_tokens': 1024,
                        'image_tokens': None,
                        'text_tokens': None
                    }
                },
                'model_provider': 'openai',
                'model_name': 'group/deepseek-flash',
                'system_fingerprint': None,
                'id': 'chatcmpl-02179473851098507ebaf30756c4961bf6a829967aba6508d40ff',
                'finish_reason': 'stop',
                'logprobs': None
            },
       